In [1]:
import numpy as np
import joblib
import pandas as pd
import requests
from datetime import timedelta

In [2]:
# ==============================================================================
# 1. Fetch prediction model
# ==============================================================================

final_rf = joblib.load("final_rf.joblib")

model_features = list(final_rf.feature_names_in_)
#print("Expected features:", model_features)

Expected features: ['temp_max', 'temp_min', 'temp_mean', 'humidity_mean', 'dewpoint_mean', 'pressure_mean', 'cloud_cover_mean', 'wind_speed_max', 'precipitation_today', 'temp_range', 'dewpoint_spread', 'pressure_change', 'month_cos', 'month_sin', 'weather_cluster']


In [3]:
# ==============================================================================
# 2. Fetch Recent Daily Weather Inputs
# ==============================================================================

# Use the same coordinates, timezone and weather variables
# as the original training-data pipeline.
LATITUDE = 55.71
LONGITUDE = 13.19
TIMEZONE = "Europe/Berlin"

# The model predicts the current calendar day's precipitation
# using the previous complete day's weather features.
local_now = pd.Timestamp.now(tz=TIMEZONE)

target_date = local_now.date()
feature_date = target_date - timedelta(days=1)
previous_date = feature_date - timedelta(days=1)

print("Prediction target date:", target_date)
print("Input feature date:", feature_date)
print("Previous date for pressure change:", previous_date)

Prediction target date: 2026-10-18
Input feature date: 2026-10-17
Previous date for pressure change: 2026-10-16


In [4]:
# Request the previous two complete calendar days
url = "https://archive-api.open-meteo.com/v1/archive"

hourly_variables = [
    "temperature_2m",
    "relative_humidity_2m",
    "dew_point_2m",
    "precipitation",
    "surface_pressure",
    "cloud_cover",
    "wind_speed_10m",
]

params = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "start_date": previous_date.isoformat(),
    "end_date": feature_date.isoformat(),
    "hourly": hourly_variables,
    "timezone": TIMEZONE,
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()

data = response.json()

if data.get("error"):
    raise RuntimeError(
        data.get("reason", "Open-Meteo API returned an error.")
    )

df_hourly_live = pd.DataFrame(data["hourly"])

df_hourly_live["time"] = pd.to_datetime(
    df_hourly_live["time"]
)

df_hourly_live["date"] = df_hourly_live["time"].dt.date

# Check that both dates are present
hourly_counts = df_hourly_live.groupby("date").size()

print("\n=== Hourly Record Counts ===")
print(hourly_counts)

for required_date in [previous_date, feature_date]:
    if required_date not in hourly_counts.index:
        raise ValueError(
            f"Weather data for {required_date} is unavailable."
        )

    # A complete local day normally has 24 hourly records.
    # Daylight-saving transitions can produce 23 or 25.
    count = hourly_counts.loc[required_date]

    if not 23 <= count <= 25:
        raise ValueError(
            f"Incomplete hourly data for {required_date}: "
            f"{count} records."
        )

# Check for missing hourly weather values
missing_values = (
    df_hourly_live[hourly_variables]
    .isna()
    .sum()
)

print("\n=== Missing Hourly Values ===")
print(missing_values)

if missing_values.any():
    raise ValueError(
        "Missing hourly weather values detected. "
        "Prediction cancelled."
    )

print("\nWeather data retrieved and checked successfully.")
print(df_hourly_live.head())


=== Hourly Record Counts ===
date
2026-10-16    24
2026-10-17    24
dtype: int64

=== Missing Hourly Values ===
temperature_2m          0
relative_humidity_2m    0
dew_point_2m            0
precipitation           0
surface_pressure        0
cloud_cover             0
wind_speed_10m          0
dtype: int64

Weather data retrieved and checked successfully.
                 time  temperature_2m  relative_humidity_2m  dew_point_2m  \
0 2026-10-16 00:00:00            13.2                    88          11.3   
1 2026-10-16 01:00:00            12.7                    92          11.4   
2 2026-10-16 02:00:00            13.3                    90          11.7   
3 2026-10-16 03:00:00            13.7                    89          11.9   
4 2026-10-16 04:00:00            13.1                    91          11.7   

   precipitation  surface_pressure  cloud_cover  wind_speed_10m        date  
0            0.0            1014.1           99            14.6  2026-10-16  
1            0.0       

In [5]:
#print(df_hourly_live)

In [6]:
# ==============================================================================
# 3. Convert Hourly Weather Data into Daily Features
# ==============================================================================

# Aggregate hourly measurements into daily features
df_daily_live = (
    df_hourly_live.groupby("date")
    .agg(
        temp_max=("temperature_2m", "max"),
        temp_min=("temperature_2m", "min"),
        temp_mean=("temperature_2m", "mean"),
        humidity_mean=("relative_humidity_2m", "mean"),
        dewpoint_mean=("dew_point_2m", "mean"),
        pressure_mean=("surface_pressure", "mean"),
        cloud_cover_mean=("cloud_cover", "mean"),
        wind_speed_max=("wind_speed_10m", "max"),
        precipitation_today=("precipitation", "sum"),
    )
    .reset_index()
    .sort_values("date")
    .reset_index(drop=True)
)

# Feature engineering: use the same formulas as training
df_daily_live["temp_range"] = (
    df_daily_live["temp_max"]
    - df_daily_live["temp_min"]
)

df_daily_live["dewpoint_spread"] = (
    df_daily_live["temp_mean"]
    - df_daily_live["dewpoint_mean"]
)

# Pressure change from the previous complete day
df_daily_live["pressure_change"] = (
    df_daily_live["pressure_mean"].diff()
)


df_daily_live["month"] = (
    pd.to_datetime(df_daily_live["date"]).dt.month
)
df_daily_live['month_sin'] = np.sin(2 * np.pi * df_daily_live['month'] / 12)
df_daily_live['month_cos'] = np.cos(2 * np.pi * df_daily_live['month'] / 12)

df_daily_live = df_daily_live.drop(columns=['month'])



# We need the previous day as well, so pressure_change
# can be calculated correctly for the input feature date.
if df_daily_live["pressure_change"].isna().any():
    first_date = df_daily_live["date"].iloc[0]
    df_daily_live.loc[
        df_daily_live["date"] == first_date,
        "pressure_change"
    ] = 0.0

# Select the target input date

feature_row = df_daily_live.loc[
    df_daily_live["date"] == feature_date
].copy()

if len(feature_row) != 1:
    raise ValueError(
        f"Expected one row for {feature_date}, "
        f"but found {len(feature_row)}."
    )

# Generate weather_cluster using the fitted training objects
scaler_cluster = joblib.load("scaler_cluster.joblib")
kmeans = joblib.load("kmeans_weather.joblib")

cluster_features = [
    "temp_mean",
    "temp_range",
    "humidity_mean",
    "dewpoint_spread",
    "pressure_mean",
    "pressure_change",
    "cloud_cover_mean",
    "wind_speed_max"
]

live_cluster_X = feature_row[cluster_features]

live_cluster_scaled = scaler_cluster.transform(
    live_cluster_X
)

feature_row["weather_cluster"] = kmeans.predict(
    live_cluster_scaled
)

print("Weather cluster:", feature_row["weather_cluster"].iloc[0])

# Select model features in the exact training order
live_X = feature_row[model_features].copy()

# Generate weather_cluster using the fitted training objects
scaler_cluster = joblib.load("scaler_cluster.joblib")
kmeans = joblib.load("kmeans_weather.joblib")

cluster_features = [
    "temp_mean",
    "temp_range",
    "humidity_mean",
    "dewpoint_spread",
    "pressure_mean",
    "pressure_change",
    "cloud_cover_mean",
    "wind_speed_max"
]

live_cluster_X = feature_row[cluster_features]

live_cluster_scaled = scaler_cluster.transform(
    live_cluster_X
)

feature_row["weather_cluster"] = kmeans.predict(
    live_cluster_scaled
)


# Select model features in the exact training order
live_X = feature_row[model_features].copy()

# Check for missing or infinite values

if not np.isfinite(live_X.to_numpy(dtype=float)).all():
    raise ValueError(
        "Invalid feature values detected. Prediction cancelled."
    )

print("=== Daily Features ===")
print("Feature date:", feature_date)
print(live_X.T.to_string(header=False))

print("\nInput shape:", live_X.shape)
print(
    "Feature order matches trained model:",
    list(live_X.columns) == model_features
)

Weather cluster: 1
=== Daily Features ===
Feature date: 2026-10-17
temp_max               12.500000
temp_min                7.300000
temp_mean               9.875000
humidity_mean          83.916667
dewpoint_mean           7.250000
pressure_mean        1015.154167
cloud_cover_mean       85.541667
wind_speed_max         19.200000
precipitation_today     3.200000
temp_range              5.200000
dewpoint_spread         2.625000
pressure_change         3.125000
month_cos               0.500000
month_sin              -0.866025
weather_cluster         1.000000

Input shape: (1, 15)
Feature order matches trained model: True


In [7]:
# ==============================================================================
# 4. Predict Next-Day Precipitation
# ==============================================================================

# 1. Predict the precipitation class
prediction = int(final_rf.predict(live_X)[0])

# 2. Get the estimated probability of precipitation
probabilities = final_rf.predict_proba(live_X)[0]

# Find the position of class 1 to ensure the probability is correct
class_1_index = list(final_rf.classes_).index(1)
precipitation_probability = probabilities[class_1_index]

# 3. Display the results
print("=" * 45)
print("       LUND WEATHER PREDICTION")
print("=" * 45)

print(f"Prediction target date: {target_date}")
print(f"Input weather date: {feature_date}")

if prediction == 1:
    print("Prediction: Precipitation expected")
else:
    print("Prediction: No significant precipitation expected")

print(
    f"Estimated probability of precipitation: "
    f"{precipitation_probability:.1%}"
)

print("Classification threshold: precipitation > 0.1 mm")
print("=" * 45)

       LUND WEATHER PREDICTION
Prediction target date: 2026-10-18
Input weather date: 2026-10-17
Prediction: Precipitation expected
Estimated probability of precipitation: 55.9%
Classification threshold: precipitation > 0.1 mm
